## Лабораторная работа: симметричное шифрование — **ноутбук преподавателя**

Этот файл зеркалирует структуру **`lab_symmetric_ciphers_ru.ipynb`** и содержит **готовые реализации** всех функций, демонстрационные ячейки и **тот же итоговый тест (20 вопросов)**.

### Назначение
- проверка заданий и демонстрация на занятии;
- эталонные ответы по сравнению с работами студентов.

### Важно
- код **учебный**, не для защиты реальных данных;
- проверки выполнены через печать **OK / ОШИБКА**, без `assert`.


### Как работать с этим ноутбуком (преподаватель)
- Выполняйте ячейки **сверху вниз**: все функции уже реализованы.
- Интерактивные ячейки с `input()` можно запускать в классе или заменить строки по месту.
- Блок **TCP**: сервер `run_echo_server()` запускайте в отдельном процессе/терминале, затем клиент.
- Итоговый тест на **ipywidgets** совпадает со студенческим ноутбуком (те же ответы).


## Подготовка: кодировка, ord и chr

Вместо «алфавита» мы будем работать с кодами символов Unicode:
- ord(c) превращает символ в целое число (кодовую точку)
- chr(i) превращает целое число в символ

В методичке предлагается выполнять операции по модулю 65536 (учебное ограничение, соответствующее диапазону BMP — первые 65536 кодовых точек Unicode). Это не «техническое» ограничение Python: `chr` в современных версиях языка может представлять и символы вне BMP, но в этой лабораторной мы специально работаем внутри BMP, чтобы сохранить единый диапазон для всех формул. Будем следовать этому правилу и считать:
- M = 65536
- Шифрование Цезаря: c = (m + k) mod M
- Дешифрование: m = (c - k) mod M

Где m и c — коды символов.


## Старт лабораторной: представьтесь

Заполните данные ниже. Мы будем использовать ваше имя в примерах, чтобы результаты выглядели «персональными».


In [ ]:
# Для демонстрации заданы значения по умолчанию. При необходимости замените строки ниже.
student_name = "Преподаватель (демо)"
group_name = "эталон"

print("Данные для заголовка теста:")
print("- Имя:", student_name)
print("- Группа:", group_name)


In [3]:
from collections import Counter
from dataclasses import dataclass
import secrets
import socket
from typing import Iterable, Optional, Tuple

UNICODE_MOD = 65536


def _wrap_u16(x: int) -> int:
    """Приводит число к диапазону [0, 65535] по модулю 65536."""
    return x % UNICODE_MOD


def _to_codes(s: str) -> list[int]:
    return [ord(ch) for ch in s]


def _from_codes(xs: Iterable[int]) -> str:
    return "".join(chr(_wrap_u16(x)) for x in xs)


def show_codes_preview(label: str, text: str, limit: int = 25) -> None:
    """Печатает первые несколько ord-кодов строки для понимания, что происходит."""
    codes = [ord(ch) for ch in text]
    head = codes[:limit]
    tail = (" ..." if len(codes) > limit else "")
    print(label)
    print("text:", text)
    print("codes:", head, tail)


> **Эталон:** функции `caesar_encrypt` / `caesar_decrypt` реализованы в следующей ячейке.

## Задание 1. Обобщённый шифр Цезаря

### Что это за алгоритм
Обобщённый шифр Цезаря — это сдвиг каждого символа на одно и то же число k.
Мы работаем не с «алфавитом», а с кодами Unicode:
- m_i = ord(символ)
- c_i = (m_i + k) mod 65536
- обратно: m_i = (c_i - k) mod 65536

Почему mod 65536: это учебное ограничение (BMP), чтобы после преобразований значения оставались в одном и том же диапазоне, который мы используем в лабораторной модели.

### Что нужно сделать
Напишите две функции:
- caesar_encrypt(k, m): int × str → str
- caesar_decrypt(k, c): int × str → str

### Подсказки (пошагово)
1) Превратите строку в список кодов: codes = _to_codes(text)
2) Для каждого кода примените формулу (с mod 65536)
3) Соберите строку обратно через _from_codes

### Самопроверка (идея)
После реализации должно выполняться:
decrypt(k, encrypt(k, m)) = m
Это свойство вы проверите в следующей интерактивной ячейке (без assert).


In [ ]:
def caesar_encrypt(k: int, m: str) -> str:
    """Шифрует строку m обобщённым шифром Цезаря (модуль UNICODE_MOD)."""
    codes = _to_codes(m)
    shifted = [_wrap_u16(x + k) for x in codes]
    return _from_codes(shifted)


def caesar_decrypt(k: int, c: str) -> str:
    """Расшифровывает шифртекст c тем же ключом k."""
    codes = _to_codes(c)
    restored = [_wrap_u16(x - k) for x in codes]
    return _from_codes(restored)


In [25]:
# Интерактивная проверка Цезаря (эталон)

m = f"Сообщение от {student_name or 'студента'}: Привет, мир!"
k = 1000

print("Открытый текст:")
print(m)

c = caesar_encrypt(k, m)
m2 = caesar_decrypt(k, c)

print("\nКлюч k =", k)
show_codes_preview("\nШифртекст:", c)
print("\nРасшифровка:")
print(m2)

print("\nСтатус:", "OK" if m2 == m else "ОШИБКА")


Открытый текст:
Сообщение от mohannad: Привет, мир!

Ключ k = 1000

Шифртекст:
text: ࠉࠦࠦ࠙࠱ࠝࠥࠠࠝЈࠦࠪЈѕїѐщііщьТЈࠇࠨࠠࠚࠝࠪДЈࠤࠠࠨЉ
codes: [2057, 2086, 2086, 2073, 2097, 2077, 2085, 2080, 2077, 1032, 2086, 2090, 1032, 1109, 1111, 1104, 1097, 1110, 1110, 1097, 1100, 1058, 1032, 2055, 2088]  ...

Расшифровка:
Сообщение от mohannad: Привет, мир!

Статус: OK


> **Эталон:** `caesar_crack_assuming_space` — в следующей ячейке.

## Задание 2. Взлом обобщённого Цезаря без ключа

### Идея (частотная эвристика)
Цезарь — это моноалфавитная подстановка: один и тот же символ исходного текста всегда превращается в один и тот же символ шифртекста.
Следствие: частоты символов «сохраняются», просто переименовываются.

Для достаточно длинного обычного текста самый частый символ часто — пробел.
Мы используем это как эвристику:
1) находим самый частый символ в шифртексте
2) считаем, что это «зашифрованный пробел»
3) вычисляем ключ k
4) расшифровываем

### Как вычислить ключ
Если c = (m + k) mod 65536, то для пробела:
ord(cipher_most_frequent) == (ord(' ') + k) mod 65536
Отсюда:
k == (ord(cipher_most_frequent) - ord(' ')) mod 65536

### Важные замечания
- Метод надёжнее на тексте примерно от 10–15 слов и больше.
- На коротком тексте самый частый символ может оказаться не пробелом.
- Если пробелов мало (например, случайные данные), эвристика не работает.

### Подсказка для улучшения (необязательно)
Попробуйте брать не 1 символ, а топ‑3 самых частых и проверять, какой результат выглядит «как текст» (например, много букв и пробелов).


In [ ]:
@dataclass(frozen=True)
class CaesarCrackResult:
    """Результат эвристического взлома Цезаря."""

    key: int
    plaintext: str
    most_frequent_cipher_char: str


def caesar_crack_assuming_space(c: str) -> CaesarCrackResult:
    """Взлом по гипотезе «самый частый символ = пробел»."""
    if c == "":
        return CaesarCrackResult(0, "", "")
    top_char, _ = Counter(c).most_common(1)[0]
    k = (ord(top_char) - ord(" ")) % UNICODE_MOD
    plaintext = caesar_decrypt(k, c)
    return CaesarCrackResult(key=k, plaintext=plaintext, most_frequent_cipher_char=top_char)


In [ ]:
# Демонстрация взлома Цезаря (эвристика «самый частый символ = пробел»)

demo_plain = (
    "Это демонстрационный текст для частотного анализа. "
    "В нём достаточно пробелов, чтобы эвристика сработала надёжно. "
    "Чем длиннее текст, тем лучше."
)

demo_key = 1234
demo_cipher = caesar_encrypt(demo_key, demo_plain)
result = caesar_crack_assuming_space(demo_cipher)

print("Оригинальный ключ:", demo_key % UNICODE_MOD)
print("Оценённый ключ:", result.key % UNICODE_MOD)
print("Самый частый символ в шифртексте:", repr(result.most_frequent_cipher_char))

print("\nФрагмент восстановленного текста:")
print(result.plaintext[:200])

if result.plaintext == demo_plain:
    print("\nРезультат: OK — текст восстановлен")
else:
    print("\nРезультат: возможно ошибка (эвристика не сработала)")


> **Эталон:** XOR с повтором и OTP — в ячейках ниже.

## Задание 3. Шифр Вернама (XOR)

### Идея
В поточном шифровании часто используют XOR (исключающее ИЛИ).
В учебной записи:
- шифртекст = открытый текст XOR потока ключа

Ключевой факт (почему это удобно):
- XOR обратен сам себе: (x XOR k) XOR k = x
Значит, операция дешифрования устроена так же, как и шифрование.

### Практический нюанс
XOR строго определён для чисел/битов. Мы работаем со строками Unicode, поэтому делаем XOR над ord-кодами символов.
Чтобы не выйти за учебный диапазон BMP, после XOR применяем _wrap_u16.

### Два режима
- Повторяющийся ключ (как идея шифра Виженера, но с XOR): ключ короче текста, повторяем по кругу.
- OTP (одноразовый блокнот): ключ той же длины, что и сообщение; биты ключа должны быть независимыми и равновероятными (в идеале — истинно случайными); ключ используется ровно один раз и хранится в секрете.

### Почему повтор ключа опасен (понимание)
Если один и тот же ключ повторяется, в данных появляется периодичность — это может позволить атаки на ключ/текст.
OTP лишён этой проблемы, но требует ключ такой же длины, как сообщение.


In [ ]:
def _repeat_key_to_length(key: str, n: int) -> str:
    """Повторяет ключ до длины n."""
    if n < 0:
        raise ValueError("n must be non-negative")
    if n == 0:
        return ""
    if key == "":
        raise ValueError("key must be non-empty when n>0")
    q, r = divmod(n, len(key))
    return key * q + key[:r]


def vernam_xor_encrypt_repeating_key(key: str, plaintext: str) -> str:
    """XOR с повторяющимся ключом."""
    if plaintext == "":
        return ""
    k2 = _repeat_key_to_length(key, len(plaintext))
    out = [_wrap_u16(ord(p) ^ ord(kk)) for p, kk in zip(plaintext, k2)]
    return _from_codes(out)


def vernam_xor_decrypt_repeating_key(key: str, ciphertext: str) -> str:
    """Для XOR шифрование и дешифрование совпадают."""
    return vernam_xor_encrypt_repeating_key(key, ciphertext)


In [ ]:
def otp_generate_key(length: int) -> str:
    """OTP-ключ: случайные коды в [0, 65535]."""
    if length < 0:
        raise ValueError("length must be non-negative")
    codes = [secrets.randbelow(UNICODE_MOD) for _ in range(length)]
    return _from_codes(codes)


def otp_encrypt(key: str, plaintext: str) -> str:
    """OTP: длина ключа = длина сообщения."""
    if len(key) != len(plaintext):
        raise ValueError("OTP key must have same length as plaintext")
    out = [_wrap_u16(ord(p) ^ ord(k)) for p, k in zip(plaintext, key)]
    return _from_codes(out)


def otp_decrypt(key: str, ciphertext: str) -> str:
    return otp_encrypt(key, ciphertext)


In [ ]:
# Интерактивная проверка Vernam/XOR и OTP

m = f"Сообщение от {student_name or 'студента'}: секрет через XOR"
key = "ключ"

print("Открытый текст:")
print(m)

c = vernam_xor_encrypt_repeating_key(key, m)
mm = vernam_xor_decrypt_repeating_key(key, c)

print("\nКлюч (строка):", key)
print("Шифртекст (может выглядеть странно):")
print(c)
print("\nРасшифровка:")
print(mm)
print("\nРезультат (повторяющийся ключ):", "OK" if mm == m else "ОШИБКА")

print("\n---\nДемо OTP")
otp_k = otp_generate_key(len(m))
c2 = otp_encrypt(otp_k, m)
mm2 = otp_decrypt(otp_k, c2)
print("Длина OTP-ключа:", len(otp_k))
print("Результат OTP:", "OK" if mm2 == m else "ОШИБКА")


## Дополнительное задание A. Заменить «сложение» на XOR (идея шифра Виженера)

Если вы реализовали повторяющийся XOR‑ключ, вы уже сделали схему в духе шифра Виженера, но с XOR вместо сложения по модулю длины алфавита.

Вопросы на понимание:
- Почему при XOR фиксированной ширины (например, байт за байтом) результат остаётся в том же диапазоне и отдельное «сложение по модулю» после XOR не требуется?
- Почему операция дешифрования совпадает с операцией шифрования?


> **Эталон:** `cbc_xor_encrypt` / `cbc_xor_decrypt` — в следующей ячейке.

## Дополнительное задание B. Цепочка блоков (CBC‑идея) для потокового XOR

### Зачем
Повторяющийся ключ приводит к повторяющимся статистическим эффектам. Один из способов усложнить анализ — сделать так, чтобы каждый блок шифровался с учётом предыдущего.

### Упрощённая модель
- Блок — это кусок строки длины len(key)
- IV — случайная строка такой же длины
- Для первого блока: xored = block XOR IV, затем cipher_block = xored XOR key
- Для следующих: xored = block XOR prev_cipher_block, затем cipher_block = xored XOR key

### Формат передачи
Чтобы получатель мог расшифровать, IV обычно передают в открытом виде вместе с шифртекстом (IV не является секретом, но должен быть уникальным для данного ключа в типичных режимах). В учебном варианте удобно вернуть (iv, ciphertext) или склеить в одну строку.

### Подсказки
- Вам потребуется функция XOR двух строк одинаковой длины.
- Если последний блок короче, можно дополнить (padding).

Ниже предлагается вариант с padding пробелами до кратности длине ключа.


In [ ]:
def xor_strings_same_length(a: str, b: str) -> str:
    if len(a) != len(b):
        raise ValueError("a and b must have same length")
    return _from_codes([_wrap_u16(ord(x) ^ ord(y)) for x, y in zip(a, b)])


def pad_to_block_size(s: str, block_size: int, pad_char: str = " ") -> str:
    if block_size <= 0:
        raise ValueError("block_size must be > 0")
    if len(pad_char) != 1:
        raise ValueError("pad_char must be a single character")
    r = len(s) % block_size
    if r == 0:
        return s
    return s + pad_char * (block_size - r)


def cbc_xor_encrypt(key: str, plaintext: str, iv: Optional[str] = None) -> Tuple[str, str, int]:
    if key == "":
        raise ValueError("key must be non-empty")
    block_size = len(key)
    original_len = len(plaintext)
    pt = pad_to_block_size(plaintext, block_size)
    if iv is None:
        iv = otp_generate_key(block_size)
    if len(iv) != block_size:
        raise ValueError("iv must have length == len(key)")
    out_blocks: list[str] = []
    prev = iv
    for i in range(0, len(pt), block_size):
        block = pt[i : i + block_size]
        cipher_block = xor_strings_same_length(xor_strings_same_length(block, prev), key)
        out_blocks.append(cipher_block)
        prev = cipher_block
    return iv, "".join(out_blocks), original_len


def cbc_xor_decrypt(key: str, iv: str, ciphertext: str, original_len: int) -> str:
    if key == "":
        raise ValueError("key must be non-empty")
    block_size = len(key)
    if len(iv) != block_size:
        raise ValueError("iv must have length == len(key)")
    if len(ciphertext) % block_size != 0:
        raise ValueError("ciphertext length must be multiple of block_size")
    out_blocks: list[str] = []
    prev = iv
    for i in range(0, len(ciphertext), block_size):
        cipher_block = ciphertext[i : i + block_size]
        block = xor_strings_same_length(xor_strings_same_length(cipher_block, key), prev)
        out_blocks.append(block)
        prev = cipher_block
    return "".join(out_blocks)[:original_len]


In [ ]:
# Интерактивная проверка CBC‑идеи (учебная XOR‑цепочка)

key = "секрет"  # длина блока = len(key)
m = f"Демо CBC: сообщение от {student_name or 'студента'}"

print("Открытый текст:")
print(m)

iv, c, n = cbc_xor_encrypt(key, m)
mm = cbc_xor_decrypt(key, iv, c, n)

print("\nКлюч:", key)
print("IV:")
print(iv)
print("\nШифртекст:")
print(c)
print("\nРасшифровка:")
print(mm)
print("\nСтатус:", "OK" if mm == m else "ОШИБКА")


> **Эталон:** транспорт и TCP — реализация Цезаря с ключом `TCP_CAESAR_K`.

## Практика: обмен сообщениями через TCP (сервер эхо/клиент эхо)

### Цель
Научиться применять ваши функции шифрования/дешифрования для передачи сообщений по сети.

### Что важно знать про TCP (очень кратко)
- TCP передаёт поток байтов. Чтобы обмениваться «сообщениями», нужно договориться о границах сообщений.
- Здесь мы используем простую разметку: **одна строка = одно сообщение**, конец сообщения — символ `\n`.

### Проблема: шифртекст может быть непечатным
Результат ваших учебных шифров может содержать любые символы. Передавать такие строки напрямую по UTF‑8 неудобно.
Поэтому ниже используется простой и надёжный формат передачи:

- строку (шифртекст) превращаем в список чисел ord(...)
- отправляем числа как одну строку: "123 456 789" + символ перевода строки
- на приёме делаем обратное преобразование через chr

Это просто упаковка данных для транспорта. Криптографию вы делаете в своих функциях шифрования и дешифрования.

### Подсказка по отладке
Если что-то не работает, сначала попробуйте прогнать цепочку локально (без сети):
открытый текст -> шифрование -> упаковка -> распаковка -> дешифрование -> открытый текст


In [ ]:
def pack_text_as_decimal_codes(s: str) -> str:
    """Упаковывает строку: коды символов через пробел."""
    return " ".join(str(ord(ch)) for ch in s)


def unpack_decimal_codes_to_text(s: str) -> str:
    """Распаковывает строку вида '65 66 67' обратно в текст."""
    s = s.strip()
    if s == "":
        return ""
    nums = [int(x) for x in s.split()]
    return "".join(chr(_wrap_u16(n)) for n in nums)


def send_line(sock: socket.socket, line: str) -> None:
    data = (line + "\n").encode("utf-8")
    sock.sendall(data)


def recv_line(sock: socket.socket) -> str:
    buf = bytearray()
    while True:
        chunk = sock.recv(1)
        if chunk == b"":
            raise ConnectionError("socket closed")
        if chunk == b"\n":
            break
        buf.extend(chunk)
    return buf.decode("utf-8")


### Задание TCP‑1: сервер

Сделайте сервер эхо, который:
1) принимает от клиента упакованный шифртекст (строка чисел)
2) распаковывает и дешифрует, печатает открытый текст
3) формирует ответ (например: 'ОТВЕТ: <сообщение>')
4) шифрует ответ и отправляет обратно

Подключите ваши функции шифрования/дешифрования в местах TODO.


In [ ]:
HOST = "127.0.0.1"
PORT = 9009

# Эталон: тот же ключ и алгоритм, что в шаблоне студенческого задания (Цезарь)
TCP_CAESAR_K = 1234


def encrypt_for_transport(plaintext: str) -> str:
    ciphertext = caesar_encrypt(TCP_CAESAR_K, plaintext)
    return pack_text_as_decimal_codes(ciphertext)


def decrypt_from_transport(packed_ciphertext: str) -> str:
    ciphertext = unpack_decimal_codes_to_text(packed_ciphertext)
    return caesar_decrypt(TCP_CAESAR_K, ciphertext)


def run_echo_server() -> None:
    """Одноподключенческий сервер эхо (как в лабораторной)."""
    print(f"Ожидание подключения на {HOST}:{PORT} ...")
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as server:
        server.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
        server.bind((HOST, PORT))
        server.listen(1)
        conn, addr = server.accept()
        with conn:
            print("Подключен клиент:", addr)
            packed = recv_line(conn)
            msg = decrypt_from_transport(packed)
            print("Расшифрованное сообщение от клиента:", msg)
            reply_plain = "ОТВЕТ: " + msg
            send_line(conn, encrypt_for_transport(reply_plain))


# Запуск сервера — в отдельном процессе/терминале:
# run_echo_server()


### Задание TCP‑2: клиент

Клиент:
1) берёт открытый текст
2) шифрует и упаковывает
3) отправляет на сервер
4) принимает ответ, распаковывает и дешифрует

Важно: используйте тот же алгоритм и ключ, что и на сервере.


In [ ]:
def run_client(message: str) -> str:
    """TCP-клиент: шифрование -> отправка -> приём -> дешифрование."""
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.connect((HOST, PORT))

        packed = encrypt_for_transport(message)
        send_line(sock, packed)

        packed_reply = recv_line(sock)
        return decrypt_from_transport(packed_reply)


# Пример использования:
# 1) В отдельном процессе запустите run_echo_server()
# 2) Здесь выполните:
# print(run_client("Привет сервер"))


## Контрольные вопросы (ответьте письменно)

1) В чём заключается алгоритм частотного анализа?
- Что именно мы измеряем?
- Почему это работает для моноалфавитной подстановки?

2) Какие есть другие распространённые атаки на криптографические алгоритмы?
- Перебор ключа
- Атаки по известному открытому тексту
- Атаки по выбранному открытому тексту
- Побочные каналы (например, по времени и энергопотреблению)

3) Какие симметричные шифры используются в настоящее время и считаются надёжными?
- AES (в режимах GCM/CTR и др.)
- ChaCha20‑Poly1305

Отдельно: почему «просто AES‑CBC» без аутентификации — это не полный ответ (нужна AEAD‑схема или отдельный MAC).


## Дополнительные задания (по желанию)

### 1) OTP (одноразовый блокнот)
- Доведите otp_encrypt/otp_decrypt до идеального восстановления исходного текста.
- Сформулируйте условия абсолютной стойкости OTP.

### 2) CBC (цепочка блоков)
- Реализуйте cbc_xor_encrypt/cbc_xor_decrypt.
- Попробуйте изменить padding: например, PKCS#7‑подобный (для учебных целей).

### 3) (*) Сеть Фейстеля
Сделайте минимальную игрушечную сеть Фейстеля:
- Разбейте блок на левую и правую половины.
- На каждом раунде: L, R = R, L XOR F(R, round_key).
- Проверьте, что дешифрование выполняется теми же шагами в обратном порядке.

Подсказка: начните с маленьких блоков (например, по 8 бит) и работайте с байтами.


## Что сдавать
- Код всех реализованных функций.
- Скриншоты или вывод интерактивных проверок с пометкой OK/ОШИБКА.
- Короткое описание эксперимента: на каких текстах взлом Цезаря сработал/не сработал и почему.
- Краткий комментарий по TCP-части: какой алгоритм выбран для транспорта и почему.


## Мини-анализ результатов (простые вопросы студенту)

После выполнения интерактивных проверок ответьте коротко:
1) Как изменился шифртекст при смене ключа Цезаря (например, k=10 и k=1000)?
2) На каком тексте частотный взлом сработал лучше: коротком или длинном? Почему?
3) Что будет, если в OTP взять ключ не той длины?
4) Сравните CBC и повторяющийся XOR: где визуально меньше повторяющихся паттернов?
5) Для TCP-обмена какой алгоритм вы выбрали и почему?


In [ ]:
# Сравнение двух ключей Цезаря на одном тексте (можно задать вручную)

sample = input("Введите короткий текст для эксперимента: ").strip() or "Пример текста"

try:
    k_small = int(input("Ключ Цезаря #1 (например 10): ").strip() or "10")
    k_big = int(input("Ключ Цезаря #2 (например 1000): ").strip() or "1000")

    c1 = caesar_encrypt(k_small, sample)
    c2 = caesar_encrypt(k_big, sample)

    print("\nОткрытый текст:", sample)
    show_codes_preview(f"\nЦезарь k={k_small}", c1)
    show_codes_preview(f"\nЦезарь k={k_big}", c2)

    same = c1 == c2
    print("\nСовпадают ли два шифртекста при разных ключах?", "Да" if same else "Нет")
    print("Вывод: при разных ключах шифртексты должны различаться.")
except ValueError:
    print("Ключи должны быть целыми числами")


## Методические подсказки (без готового кода)

### Типичные ошибки и как их избежать
- **Забыли модуль 65536**: при Цезаре в этой модели нарушается согласованность с учебным диапазоном BMP и формулами лабораторной.
- **Перепутали направление в дешифровании**: для Цезаря нужно вычитать ключ.
- **Разная логика в шифровании и дешифровании XOR**: для XOR операция симметрична (шифрование и дешифрование выполняются одинаково).
- **OTP с ключом другой длины**: для OTP длина ключа должна совпадать с длиной сообщения.
- **CBC без корректной длины IV**: длина IV должна быть равна длине ключа (размеру блока в этой лабораторной модели).

### Мини-план отладки
1) Сначала проверьте одну букву/один символ руками.
2) Потом проверьте короткую строку из 3-5 символов.
3) Только затем переходите к длинным сообщениям.
4) Для TCP сначала проверьте цепочку локально: открытый текст -> шифрование -> упаковка -> распаковка -> дешифрование.

### Как понять, что реализация корректна
- Расшифрованный текст полностью совпадает с исходным.
- При изменении ключа меняется и шифртекст.
- Для частотного взлома длинный текст обычно восстанавливается лучше короткого.


## Итоговый тест: 20 вопросов (эталон совпадает со студенческим ноутбуком)

Ниже — **те же вопросы и правильные индексы ответов**, что в `lab_symmetric_ciphers_ru.ipynb`. Используйте для проверки MCQ.


In [ ]:
# Интерактивный тест (20 вопросов)
# Нужен пакет ipywidgets (обычно доступен в JupyterLab)

import ipywidgets as widgets
from IPython.display import display, HTML

questions = [
    {
        "q": "1) Что характеризует симметричное шифрование?",
        "options": [
            "Один и тот же ключ используется для шифрования и расшифрования",
            "Всегда используются два разных ключа",
            "Ключ хранится только у сервера",
            "Секретность не зависит от ключа",
        ],
        "answer": 0,
    },
    {
        "q": "2) Основная слабость моноалфавитной подстановки (например, Цезарь):",
        "options": [
            "Нельзя расшифровать даже с ключом",
            "Не работает с цифрами",
            "Сохраняется частотная структура текста",
            "Слишком большой ключ",
        ],
        "answer": 2,
    },
    {
        "q": "3) Для обобщённого Цезаря в этой лабораторной используется модуль:",
        "options": ["26", "255", "65536", "1114112"],
        "answer": 2,
    },
    {
        "q": "4) Что возвращает функция ord(c) в Python?",
        "options": [
            "Строку из одного символа",
            "Целочисленный код символа",
            "Бинарный ключ",
            "Хэш символа",
        ],
        "answer": 1,
    },
    {
        "q": "5) Какое свойство XOR позволяет использовать одну и ту же операцию для шифрования и дешифрования?",
        "options": [
            "Коммутативность (перестановка слагаемых)",
            "Инволютивность: (x XOR k) XOR k = x",
            "XOR всегда увеличивает число",
            "XOR требует модуль",
        ],
        "answer": 1,
    },
    {
        "q": "6) OTP (одноразовый блокнот) теоретически стойкий, если:",
        "options": [
            "Ключ короче сообщения и повторяется",
            "Ключ такой же длины, что и сообщение; биты ключа независимы и равновероятны; ключ используется один раз и хранится в секрете",
            "Ключ хранится в открытом виде",
            "Используется только латиница",
        ],
        "answer": 1,
    },
    {
        "q": "7) Почему повторяющийся ключ в XOR-схеме опасен?",
        "options": [
            "Появляются статистические и периодические зависимости",
            "Нельзя выполнить дешифрование",
            "Ключ становится длиннее сообщения",
            "Невозможно передать данные по сети",
        ],
        "answer": 0,
    },
    {
        "q": "8) Что обычно передают вместе с шифртекстом в CBC-подобных схемах?",
        "options": ["Секретный ключ шифрования", "IV (вектор инициализации)", "Пароль пользователя", "Хэш пароля"],
        "answer": 1,
    },
    {
        "q": "9) Какова роль IV?",
        "options": [
            "Заменяет ключ",
            "Гарантирует, что один и тот же открытый текст всегда даст один и тот же шифртекст при том же ключе",
            "Снижает повторяемость паттернов между сообщениями",
            "Ускоряет дешифрование за счёт удаления ключа",
        ],
        "answer": 2,
    },
    {
        "q": "10) Частотный анализ обычно работает лучше всего, когда:",
        "options": [
            "Текст достаточно длинный",
            "Текст состоит из 2–3 символов",
            "Ключ очень маленький",
            "Используется только ASCII",
        ],
        "answer": 0,
    },
    {
        "q": "11) Какой из перечисленных вариантов относится к AEAD (аутентифицированному шифрованию)?",
        "options": ["AES-ECB", "AES-CBC без аутентификации (без MAC/AEAD)", "AES-GCM", "ROT13"],
        "answer": 2,
    },
    {
        "q": "12) Почему AES-ECB считается небезопасным для многих задач?",
        "options": [
            "Не поддерживает ключи",
            "Одинаковые блоки открытого текста дают одинаковые блоки шифртекста",
            "Слишком медленный",
            "Требует интернет",
        ],
        "answer": 1,
    },
    {
        "q": "13) Что обязательно для корректного OTP в этой лабораторной?",
        "options": [
            "len(key) == len(message)",
            "len(key) < len(message)",
            "Ключ должен быть строкой 'ключ'",
            "Ключ должен состоять только из цифр",
        ],
        "answer": 0,
    },
    {
        "q": "14) В модели CBC из лабораторной размер блока равен:",
        "options": [
            "1 байт",
            "len(key)",
            "len(message)",
            "всегда 128 бит",
        ],
        "answer": 1,
    },
    {
        "q": "15) Что верно про симметричные алгоритмы в целом?",
        "options": [
            "Они всегда медленнее асимметричных",
            "Они обычно быстрее асимметричных и подходят для шифрования данных",
            "Они не используют ключи",
            "Они подходят только для цифровых подписей",
        ],
        "answer": 1,
    },
    {
        "q": "16) Что делает функция chr(i) в Python?",
        "options": [
            "Возвращает код символа",
            "Возвращает символ по числовому коду",
            "Шифрует строку",
            "Делает XOR",
        ],
        "answer": 1,
    },
    {
        "q": "17) Что критично при TCP-обмене зашифрованными сообщениями?",
        "options": [
            "Игнорировать границы сообщений",
            "Определить формат фрейминга (например, строки с \n)",
            "Передавать только латинские символы",
            "Не декодировать байты",
        ],
        "answer": 1,
    },
    {
        "q": "18) Если дешифрование не восстанавливает исходный текст, сначала стоит проверить:",
        "options": [
            "Одинаковый ли ключ/алгоритм используется на обеих сторонах",
            "Размер экрана в Jupyter",
            "Скорость интернета",
            "Цвет темы IDE",
        ],
        "answer": 0,
    },
    {
        "q": "19) Какой современный вариант потокового шифрования + аутентификации широко используется?",
        "options": ["ChaCha20-Poly1305", "Шифр Виженера", "Шифр Цезаря", "DES в режиме ECB"],
        "answer": 0,
    },
    {
        "q": "20) Главное педагогическое ограничение этой лабораторной:",
        "options": [
            "Рассматриваются учебные модели, а не промышленные криптосистемы",
            "Нельзя использовать Python",
            "Нельзя запускать код",
            "Все алгоритмы абсолютно стойкие",
        ],
        "answer": 0,
    },
]

score = {"correct": 0, "answered": 0}
question_states = [False] * len(questions)

student_label = student_name if str(student_name).strip() else "Студент"
group_label = group_name if str(group_name).strip() else "(группа не указана)"

header = widgets.HTML("<h3 style='margin:0;'>Тест (эталон преподавателя)</h3>")
progress = widgets.HTML("")


def update_progress():
    total = len(questions)
    correct = score["correct"]
    answered = score["answered"]
    percent = (100.0 * correct / total) if total else 0.0

    status = ""
    if answered == total:
        status = "<span style='color:#0a7f2e;'><b>Тест завершён.</b></span>"

    progress.value = (
        f"<b>Студент:</b> {student_label} | <b>Группа:</b> {group_label}<br>"
        f"<b>Результат:</b> {correct} / {total} ({percent:.1f}%) | "
        f"<b>Отвечено:</b> {answered} / {total}<br>{status}"
    )


def make_question_widget(q_idx, item):
    title = widgets.HTML(f"<b>{item['q']}</b>")
    feedback = widgets.HTML("<span style='color:#555;'>Выберите один вариант.</span>")

    buttons = []
    box = widgets.VBox()

    def on_click_factory(opt_idx):
        def _handler(btn):
            if question_states[q_idx]:
                return

            question_states[q_idx] = True
            score["answered"] += 1

            correct_idx = item["answer"]
            for i, b in enumerate(buttons):
                b.disabled = True
                if i == correct_idx:
                    b.button_style = "success"
                if i == opt_idx and opt_idx != correct_idx:
                    b.button_style = "danger"

            if opt_idx == correct_idx:
                score["correct"] += 1
                feedback.value = "<span style='color:green;'><b>Верно.</b></span>"
            else:
                correct_text = item["options"][correct_idx]
                feedback.value = (
                    "<span style='color:red;'><b>Неверно.</b></span> "
                    f"<span style='color:#333;'>Правильный ответ: {correct_text}</span>"
                )

            update_progress()
        return _handler

    for i, opt in enumerate(item["options"]):
        b = widgets.Button(
            description=opt,
            button_style="",
            layout=widgets.Layout(width="95%"),
            style={"button_color": None},
        )
        b.on_click(on_click_factory(i))
        buttons.append(b)

    box.children = [title] + buttons + [feedback, widgets.HTML("<hr style='margin:10px 0;'>")]
    return box


all_questions = [make_question_widget(i, q) for i, q in enumerate(questions)]
quiz_ui = widgets.VBox([header, progress] + all_questions)

update_progress()
display(quiz_ui)


## Чек-лист для преподавателя

| Тема | Эталон в ноутбуке |
|------|-------------------|
| Цезарь mod 65536 | `caesar_encrypt` / `caesar_decrypt` |
| Частотный взлом | `caesar_crack_assuming_space` |
| XOR повтор ключа | `vernam_xor_*` |
| OTP | `otp_generate_key`, `otp_encrypt` / `otp_decrypt` |
| CBC-подобный XOR | `cbc_xor_encrypt` / `cbc_xor_decrypt` |
| TCP | `encrypt_for_transport` / `decrypt_for_transport`, `TCP_CAESAR_K = 1234`, ответ `ОТВЕТ:` |
| MCQ | 20 вопросов, индексы ответов как в студенческом файле |

При проверке работ сравнивайте корректность инварианта `decrypt(k, encrypt(k, m)) == m` и согласованность ключа в TCP.
